In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

root = Path.cwd()
if not (root / 'pipeline_config.yaml').exists() and (root.parent / 'pipeline_config.yaml').exists():
    root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Working directory: {root}')

Working directory: c:\Users\Usuario\OneDrive\Escritorio\preProcessing


# Phase 6 - Baseline Modeling

This notebook trains minimal baseline regressors to validate whether the integrated feature matrix carries predictive signal for next-window volatility.

In [2]:
from pathlib import Path

import pandas as pd

from run_pipeline import load_config
from src.modeling.baseline import plot_top_feature_importance, run_baseline_suite, summarize_results

In [3]:
config = load_config()
modeling_cfg = config.get('modeling', {})
max_train_rows = int(modeling_cfg.get('max_train_rows', 250_000))
random_state = int(modeling_cfg.get('random_state', 42))

artifacts = run_baseline_suite(
    config=config,
    max_train_rows=max_train_rows,
    random_state=random_state,
)

results_df = artifacts.results.copy()
importance_df = artifacts.feature_importance.copy()
print('Baselines executed:', len(results_df))

Baselines executed: 3


In [4]:
results_df = results_df.sort_values('validation_mae').reset_index(drop=True)
results_path = Path('reports/modeling_baseline_results.csv')
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)

summary = summarize_results(results_df)
print('Saved baseline results to', results_path)
print('Summary metrics:', summary)
results_df

Saved baseline results to reports\modeling_baseline_results.csv
Summary metrics: {'XGBoostRegressor_price_plus_sentiment_validation_mae': 0.1602995640540117, 'LinearRegression_price_plus_sentiment_validation_mae': 0.1674502961223491, 'LinearRegression_price_only_validation_mae': 0.16745186790923935, 'XGBoostRegressor_price_plus_sentiment_test_mae': 0.1597641816216894, 'LinearRegression_price_plus_sentiment_test_mae': 0.17547488020706478, 'LinearRegression_price_only_test_mae': 0.17552368308529775, 'linear_validation_improvement_pct': 0.0009386499594648256}


,model,feature_set,n_features,train_rows_used,validation_mae,test_mae,status
0,XGBoostRegressor_price_plus_sentiment,price_plus_sentiment,33,250000,0.160300,0.159764,ok
1,LinearRegression_price_plus_sentiment,price_plus_sentiment,33,250000,0.167450,0.175475,ok
2,LinearRegression_price_only,price_only,25,250000,0.167452,0.175524,ok


In [5]:
importance_plot_path = Path('reports/plots/modeling_top20_features.png')
top_features_df = plot_top_feature_importance(
    feature_importance=importance_df,
    output_path=importance_plot_path,
    top_n=20,
)
print('Saved feature importance plot to', importance_plot_path)

Saved feature importance plot to reports\plots\modeling_top20_features.png


In [6]:
top_features_df

,model,feature,importance
77,XGBoostRegressor_price_plus_sentiment,rsi_14,0.013054
76,XGBoostRegressor_price_plus_sentiment,finbert_pct_negative,0.013189
75,XGBoostRegressor_price_plus_sentiment,log_return_lag_5d,0.013729
74,XGBoostRegressor_price_plus_sentiment,rolling_mean_60d,0.015632
73,XGBoostRegressor_price_plus_sentiment,bb_width,0.016448
72,XGBoostRegressor_price_plus_sentiment,log_return_lag_3d,0.018132
71,XGBoostRegressor_price_plus_sentiment,rolling_mean_5d,0.018140
70,XGBoostRegressor_price_plus_sentiment,bb_upper,0.018317
69,XGBoostRegressor_price_plus_sentiment,log_return_lag_1d,0.018830
68,XGBoostRegressor_price_plus_sentiment,realized_vol_5d,0.021432
